# Stability of psychopathology profiles
Maria B. Jelen

This script tests the overall stability of individuals' distance to psychopathology profiles on SOM topology. At every follow-up time point (1, 2, 3, 4 years), each participant's BMU on the SOM is recalculated, and a vector of Euclidean distances to each profile's centroid in feature space is calculated. This is then correlated with the baseline distance vector to establish general stability of proximity to different prsychopathology profiles.

In [1]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom 

In [2]:
# Plotting parameters
plt.rcParams.update({
    "font.family": "Arial",
    "font.weight": "normal"
})

In [3]:
with open('som.p', 'rb') as infile:
    som = pickle.load(infile)

### Assessing stability across measurement timepoints
- Load data from other timepoints
- Rematch to SOM (obtain BMUs)
- Calculate each individual's distance to psychopathology profile centroids
- Calculate correlation of distance to centroids vector with baseline distance vector

In [4]:
# Load CSV file
baseline = pd.read_csv('mock_baseline.csv') 
print(baseline.shape)

(200, 11)


In [5]:
data_1year = pd.read_csv('mock_1year.csv') 
data_2year = pd.read_csv('mock_2year.csv') 
data_3year = pd.read_csv('mock_3year.csv') 
data_4year = pd.read_csv('mock_4year.csv') 

In [6]:
# Check participant numbers across years
all_dfs = [baseline, data_1year, data_2year, data_3year, data_4year]

for df in all_dfs:
    print(df.shape)

(200, 11)
(200, 11)
(200, 11)
(200, 11)
(200, 11)


In [7]:
# Set variable order
new_order = [
    'Participant_ID', 'cbcl_scr_syn_anxdep_r',
    'cbcl_scr_syn_withdep_r', 'cbcl_scr_syn_somatic_r',
    'cbcl_scr_syn_social_r', 'cbcl_scr_syn_thought_r',
    'cbcl_scr_syn_attention_r', 'cbcl_scr_syn_rulebreak_r',
    'cbcl_scr_syn_aggressive_r',
    'sds_p_ss_dims_final',  'Mania_Total'
]

data_1year = data_1year[new_order] 
data_2year = data_2year[new_order]
data_3year = data_3year[new_order]
data_4year = data_4year[new_order]


In [8]:
# Define features to be extracted

feature_cols = ['cbcl_scr_syn_anxdep_r',
    'cbcl_scr_syn_withdep_r', 'cbcl_scr_syn_somatic_r',
    'cbcl_scr_syn_social_r', 'cbcl_scr_syn_thought_r',
    'cbcl_scr_syn_attention_r', 'cbcl_scr_syn_rulebreak_r',
    'cbcl_scr_syn_aggressive_r',
    'sds_p_ss_dims_final',  'Mania_Total'
]

# Extract baseline features
baseline_features = baseline[feature_cols].copy()
baseline_features.index = baseline['Participant_ID'].astype(str)

# Fit scaler on baseline
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(baseline_features.values) 
X_baseline_scaled = scaler.transform(baseline_features.values)


# Wrap function to extract features and scale next timepoints
timepoints = [data_1year, data_2year, data_3year, data_4year]

scaled_features = {}

for i, df in enumerate(timepoints, start=1):

    df_features = df[feature_cols].copy()
    df_features.index = df['Participant_ID'].astype(str)

    df_scaled = scaler.transform(df_features.values)

    scaled_features[f"year{i}"] = {
        "features": df_features,
        "scaled": df_scaled
    }
    

In [9]:
# new code to get distances in weight feature space instead of BMUS - for PLS distances and for stability analysis
from scipy.spatial import distance
import os

som_weights = som.get_weights()

import json
report_filename = 'significant_islands_report_mock.json'
with open(report_filename, 'r') as f:
    final_report = json.load(f)

island_weight_centroids = {}

for profile_name, islands in final_report.items():
    island_weight_centroids[profile_name] = [] # initiate list for each profile
    for island in islands:
        # island = list of row/col coords on SOM, take weight vectors
        weights = np.array([som_weights[row,col] for row, col in island])
        # define new centroids - mean of weight vectors gives centroid in feature space
        centroid = np.mean(weights, axis=0)
        island_weight_centroids[profile_name].append(centroid) # append each profile with its new centroid

with open('island_weight_centroids_mock.pkl', 'wb') as f:
    pickle.dump(island_weight_centroids, f)

print("File saved. Size:", os.path.getsize('island_weight_centroids_mock.pkl'))

print(island_weight_centroids)

File saved. Size: 397
{'simulated_externalising': [array([-0.76048979, -0.58127075, -0.65667575, -0.57760852, -0.66446338,
       -0.59287861, -0.15725324, -0.37418613, -0.56779549, -0.20228961])], 'simulated_noproblems': [array([-0.69186506, -0.71961438, -0.75756066, -0.67116683, -0.72996816,
       -0.67206339, -0.70678647, -0.7614919 , -0.55642543, -0.47344599])]}


In [10]:
# Load island centroids calculated before
from scipy.spatial import distance
import pandas as pd

with open('island_weight_centroids_mock.pkl', 'rb') as f:
    island_weight_centroids = pickle.load(f)

# print(island_weight_centroids)


In [11]:
# Get bmus for baseline
som_weights = som.get_weights()

baseline_bmus = {} 

for pid, row in zip(baseline_features.index.astype(str), X_baseline_scaled):
    bmu = som.winner(row)  # returns (row, col) coordinate
    baseline_bmus[pid] = bmu


baseline_bmu_weights = {}
for id, (row,col) in baseline_bmus.items():
    baseline_bmu_weights[id] = som_weights[row, col]

#print(len(baseline_bmus))
#print(baseline_bmu_weights)


In [12]:
# Calculate feature weight distance to island centroids
baseline_distances = []

for participant_id, bmu_weight in baseline_bmu_weights.items():
    distances = {}
    for profile_name, centroids in island_weight_centroids.items():
        # Skip no problems which has no centroids
        #if len(centroids) == 0:
           # continue

        dists = [
            distance.euclidean(bmu_weight, centroid)
            for centroid in centroids]

        distances[profile_name] = min(dists)

    distances['Participant_ID'] = participant_id
    baseline_distances.append(distances)

baseline_distances_df = pd.DataFrame(baseline_distances)

# sub id to first column
id_col = baseline_distances_df.pop('Participant_ID')
baseline_distances_df.insert(0, 'Participant_ID', id_col)

#print(baseline_distances_df.head())


In [13]:
from scipy.spatial import distance

# Dictionary to store BMUs and distances for each timepoint
bmus_dict = {}
bmu_weights_dict = {}
distances_dict = {}
distances_df_dict = {}

for tp, data in scaled_features.items():
    print(f"Processing {tp} ")

    X_scaled = data['scaled']
    participant_ids = data['features'].index.astype(str)

    # Compute BMUs
    bmus = {}
    for pid, row in zip(participant_ids, X_scaled):
        bmu = som.winner(row)
        bmus[pid] = bmu
    bmus_dict[tp] = bmus

    # Extract weight vectors for each BMU
    bmu_weights = {}
    for pid, (row, col) in bmus.items():
        bmu_weights[pid] = som_weights[row, col]
    bmu_weights_dict[tp] = bmu_weights

    # Compute distances to profile centroids
    distances_list = []
    for pid, bmu_weight in bmu_weights.items():
        distances = {}
        for profile_name, centroids in island_weight_centroids.items():
            if len(centroids) == 0:
                continue  # skip diffuse/no-centroid profiles
            dists = [distance.euclidean(bmu_weight, centroid) for centroid in centroids]
            distances[profile_name] = min(dists)
        distances['Participant_ID'] = pid
        distances_list.append(distances)

    distances_df = pd.DataFrame(distances_list)

    # Put participant ID as first column
    id_col = distances_df.pop('Participant_ID')
    distances_df.insert(0, 'Participant_ID', id_col)

    distances_df_dict[tp] = distances_df

    print(f"{tp} done: {len(bmus)} participants, {distances_df.shape[1]-1} profiles")



Processing year1 
year1 done: 200 participants, 2 profiles
Processing year2 
year2 done: 200 participants, 2 profiles
Processing year3 
year3 done: 200 participants, 2 profiles
Processing year4 
year4 done: 200 participants, 2 profiles


In [14]:
# Plot baseline correlation distribution

baseline_df = baseline_distances_df.set_index('Participant_ID')
year1_df = distances_df_dict['year1'].set_index('Participant_ID')

# Align participants (intersection)
common_ids = baseline_df.index.intersection(year1_df.index)
aligned_baseline = baseline_df.loc[common_ids]
aligned_year1 = year1_df.loc[common_ids]

# Compute row-wise correlation across profiles for each participant
baseline_year1_similarity = aligned_baseline.corrwith(aligned_year1, axis=1)
baseline_year1_similarity_abs = baseline_year1_similarity.abs()

print(baseline_year1_similarity.head(20))
print(baseline_year1_similarity.shape)

# With only 2 profiles, correlation can only take on values of -1 or 1 because the correlation is calculated from only 2 paired distances.
# With more profiles, more informative assessment of the relative distance to profiles pattern of each participant can be plotted. 

Participant_ID
P01    1.0
P02    1.0
P03   -1.0
P04    1.0
P05   -1.0
P06   -1.0
P07    1.0
P08    1.0
P09    1.0
P10    1.0
P11    1.0
P12    1.0
P13    1.0
P14    1.0
P15    1.0
P16    1.0
P17   -1.0
P18    1.0
P19    1.0
P20   -1.0
dtype: float64
(200,)


In [15]:
# code can be repeated for 2-4 year follow-up datasets